# Deep Agents 301: Under the Hood

The 201 taught you how to **use** Deep Agents. This notebook teaches you how they **work** — and how to customize the parts that matter.

**What you'll learn:**
- How `create_deep_agent()` is really just `create_agent()` + middleware
- How to build your own filesystem interface (the pattern, not the full implementation)
- How to replace the default `task()` tool with structured task definitions
- How to plug a deterministic `StateGraph` workflow in as a subagent
- How MCP adapters make external tools interchangeable with native ones
- How to use sandboxes for code execution
- What happens when context management fails — and how to fix it

**Prerequisites:** Complete the 201 Deep Agents notebook first.

## Part 0: Setup

In [ ]:
# Add project root to Python path
import sys
from pathlib import Path
project_root = Path().resolve().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.models import model
print(f"Model: {model.__class__.__name__}")

In [ ]:
# Shared imports for the notebook
from langchain_core.tools import tool
from langsmith import uuid7
from langgraph.checkpoint.memory import MemorySaver

from tavily import TavilyClient
tavily_client = TavilyClient()

@tool(parse_docstring=True)
def tavily_search(query: str) -> str:
    """Search the web for information on a given query.
    
    Args:
        query: Search query to execute.
    """
    results = tavily_client.search(query, max_results=3)
    return "\n\n".join(
        f"**{r['title']}**\n{r['url']}\n{r['content']}" 
        for r in results["results"]
    )

---
## Part 1: Deconstructing `create_deep_agent`

`create_deep_agent()` is **not** a separate agent architecture. It's a convenience wrapper around `create_agent()` that assembles a specific middleware stack.

Here's what it does (simplified):

```python
def create_deep_agent(model, tools, system_prompt, ...):
    middleware = [
        TodoListMiddleware(),                    # Planning
        FilesystemMiddleware(backend=backend),   # File tools
        SubAgentMiddleware(subagents=...),       # task() tool
        SummarizationMiddleware(model, ...),     # Context management
        PatchToolCallsMiddleware(),              # Fix dangling tool calls
    ]
    return create_agent(model, tools=tools, middleware=middleware)
```

That's it. No special graph, no custom nodes. Just middleware.

### Why this matters

If you can build the middleware stack yourself, you can:
- Swap out any piece (e.g., your own filesystem, your own task tool)
- Add middleware in different order
- Use `create_agent()` directly without the deep agents dependency

In [ ]:
# Let's build the same thing from scratch using create_agent
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from deepagents.middleware.filesystem import FilesystemMiddleware
from deepagents.middleware.summarization import create_summarization_middleware
from deepagents.middleware.patch_tool_calls import PatchToolCallsMiddleware
from deepagents.backends import StateBackend

# This is the core of what create_deep_agent does
hand_built_agent = create_agent(
    model,
    tools=[tavily_search],
    system_prompt="You are a helpful research assistant.",
    middleware=[
        TodoListMiddleware(),
        FilesystemMiddleware(backend=StateBackend),
        create_summarization_middleware(model, StateBackend),
        PatchToolCallsMiddleware(),
    ],
).with_config({"recursion_limit": 1000})

print("Agent built with create_agent + middleware")
print(f"Type: {type(hand_built_agent).__name__}")

In [ ]:
# Test it -- it behaves exactly like create_deep_agent
config = {"configurable": {"thread_id": str(uuid7())}}

result = hand_built_agent.invoke({
    "messages": [{"role": "user", "content": "Write a file called /notes.md with 'Built from scratch!' then read it back."}]
}, config=config)

print(result["messages"][-1].content)
print("\nFiles in state:", list(result.get("files", {}).keys()))

### Middleware hooks at a glance

Each middleware can implement these hooks:

| Hook | When it runs | Common use |
|------|-------------|------------|
| `before_agent(state, runtime)` | Once at start | Load skills, memory |
| `wrap_model_call(request, handler)` | Each LLM call | Inject system prompt text |
| `wrap_tool_call(request, handler)` | Each tool call | Log, intercept, modify |
| `tools` (property) | Registration time | Add tools to the agent |
| `state_schema` (property) | Graph compilation | Extend state with new fields |

Let's see these hooks in action with a simple logging middleware:

In [ ]:
from langchain.agents.middleware import wrap_model_call, wrap_tool_call

@wrap_model_call
def inspect_model_call(request, handler):
    """See what the LLM receives on each call."""
    # request.system_message contains the full system prompt (all middleware have appended to it)
    sys_msg = request.system_message
    if sys_msg:
        # System message can be multi-block from multiple middleware
        total_chars = sum(len(b.get("text", "")) for b in sys_msg.content_blocks if isinstance(b, dict))
        print(f"\n🧠 [Model Call] System prompt: {total_chars:,} chars across {len(sys_msg.content_blocks)} blocks")
    print(f"   Tools available: {[t.name for t in request.tools]}")
    print(f"   Messages in context: {len(request.messages)}")
    return handler(request)

@wrap_tool_call
def inspect_tool_call(request, handler):
    """See what tools are called and with what args."""
    name = request.tool_call["name"]
    args = request.tool_call["args"]
    # Truncate long args for display
    display_args = {k: (v[:80] + "..." if isinstance(v, str) and len(v) > 80 else v) for k, v in args.items()}
    print(f"   🔧 {name}({display_args})")
    return handler(request)

# Add our inspect middleware to the stack
inspectable_agent = create_agent(
    model,
    tools=[tavily_search],
    system_prompt="You are a helpful assistant.",
    middleware=[
        TodoListMiddleware(),
        FilesystemMiddleware(backend=StateBackend),
        create_summarization_middleware(model, StateBackend),
        PatchToolCallsMiddleware(),
        inspect_model_call,  # Our custom middleware
        inspect_tool_call,
    ],
).with_config({"recursion_limit": 1000})

config = {"configurable": {"thread_id": str(uuid7())}}
result = inspectable_agent.invoke({
    "messages": [{"role": "user", "content": "Write a file called /hello.txt with 'hi'"}]
}, config=config)

print("\n---")
print(result["messages"][-1].content)

### Key Takeaway
- `create_deep_agent()` = `create_agent()` + a specific middleware stack
- Middleware hooks let you inspect, modify, and extend any part of the agent loop
- You can swap, reorder, or replace any middleware to customize behavior

---
## Part 2: Building a Minimal Filesystem Interface

Deep Agent's `FilesystemMiddleware` provides 7 tools (`ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`, `execute`). Rather than reimplement all of them, let's build a minimal 2-tool version to understand the **pattern**.

The key insight: **filesystem tools don't touch the filesystem directly**. They:
1. Operate through a backend abstraction (`BackendProtocol`)
2. Return `Command(update={"files": {...}})` to update LangGraph state
3. Use a custom state reducer to merge file updates

This pattern separates *where* files are stored from *how* the agent uses them.

In [ ]:
from datetime import datetime, timezone
from typing import Annotated, Any, NotRequired
from langchain.agents.middleware.types import AgentMiddleware, AgentState
from langchain.tools import ToolRuntime
from langchain_core.messages import ToolMessage
from langchain_core.tools import StructuredTool
from langgraph.types import Command
from deepagents.middleware._utils import append_to_system_message


# Step 1: Define the state reducer
# This is how LangGraph merges file updates from multiple tool calls
def file_reducer(left: dict | None, right: dict) -> dict:
    """Merge file state updates. None values = deletions."""
    result = {**(left or {})}
    for key, value in right.items():
        if value is None:
            result.pop(key, None)  # Delete
        else:
            result[key] = value    # Create/update
    return result


# Step 2: Define the state schema
class MinimalFilesystemState(AgentState):
    files: Annotated[NotRequired[dict], file_reducer]


# Step 3: Build the middleware
class MinimalFilesystemMiddleware(AgentMiddleware):
    """A stripped-down filesystem middleware to show the pattern."""
    
    state_schema = MinimalFilesystemState
    
    def __init__(self):
        super().__init__()
        self.tools = [self._make_write_tool(), self._make_read_tool()]
    
    def _make_write_tool(self):
        def write_file(
            file_path: Annotated[str, "Absolute path for the file (e.g., /notes.md)"],
            content: Annotated[str, "Content to write"],
            runtime: ToolRuntime,
        ) -> Command:
            """Write a file to the virtual filesystem."""
            now = datetime.now(timezone.utc).isoformat()
            file_data = {
                "content": content.split("\n"),
                "created_at": now,
                "modified_at": now,
            }
            # THIS is the key pattern: return a Command to update state
            return Command(update={
                "files": {file_path: file_data},
                "messages": [ToolMessage(
                    f"Wrote {len(content)} chars to {file_path}",
                    tool_call_id=runtime.tool_call_id,
                )],
            })
        
        return StructuredTool.from_function(name="write_file", func=write_file)
    
    def _make_read_tool(self):
        def read_file(
            file_path: Annotated[str, "Absolute path of the file to read"],
            runtime: ToolRuntime,
        ) -> Command:
            """Read a file from the virtual filesystem."""
            files = runtime.state.get("files", {})
            if file_path not in files:
                return Command(update={
                    "messages": [ToolMessage(
                        f"Error: {file_path} not found",
                        tool_call_id=runtime.tool_call_id,
                    )],
                })
            content = "\n".join(files[file_path]["content"])
            return Command(update={
                "messages": [ToolMessage(
                    content,
                    tool_call_id=runtime.tool_call_id,
                )],
            })
        
        return StructuredTool.from_function(name="read_file", func=read_file)
    
    def wrap_model_call(self, request, handler):
        """Tell the LLM about its filesystem capabilities."""
        instructions = """## Filesystem
You have `write_file` and `read_file` tools. Use absolute paths (e.g., `/notes.md`).
Files are stored in-memory and persist within this thread."""
        new_sys = append_to_system_message(request.system_message, instructions)
        return handler(request.override(system_message=new_sys))


print("MinimalFilesystemMiddleware defined")
print("Tools:", [t.name for t in MinimalFilesystemMiddleware().tools])

In [ ]:
# Use our minimal filesystem with create_agent
minimal_fs_agent = create_agent(
    model,
    system_prompt="You are a helpful assistant.",
    middleware=[MinimalFilesystemMiddleware()],
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = minimal_fs_agent.invoke({
    "messages": [{"role": "user", "content": "Write a file /demo.txt with 'Hello from minimal filesystem!' then read it back."}]
}, config=config)

print("Agent response:", result["messages"][-1].content)
print("\nFiles in state:", {k: v["content"] for k, v in result.get("files", {}).items()})

### How this compares to the real `FilesystemMiddleware`

Our minimal version shows the core pattern. The real implementation adds:

| Feature | Our version | Real `FilesystemMiddleware` |
|---------|-------------|----------------------------|
| Tools | `write_file`, `read_file` | + `ls`, `edit_file`, `glob`, `grep`, `execute` |
| Storage | Direct state access | `BackendProtocol` abstraction |
| Pagination | No | `offset`/`limit` on reads |
| Images | No | Multimodal content blocks |
| Sandboxing | No | `SandboxBackendProtocol` |
| Path validation | No | Sanitization, virtual paths |

The **pattern** is identical: middleware registers tools, tools return `Command(update={...})`, state reducer merges updates.

### Key Takeaway
- Filesystem tools modify state via `Command(update={"files": {...}})`, not by writing to disk
- A custom state reducer (`file_reducer`) handles merging and deletions
- `wrap_model_call` injects instructions so the LLM knows what tools are available
- The real implementation adds a backend abstraction so storage is pluggable

---
## Part 3: Backend Implementations — Where Files Actually Live

In Part 2 we built a filesystem that stores files in LangGraph state. But that's just one option. Deep Agents has three backend implementations, each with fundamentally different storage semantics.

The key design: **tools don't know where files are stored**. They call `backend.write()` and the backend decides whether to update state, write to disk, or call a remote store.

### The `WriteResult` branching pattern

Every backend's `write()` returns a `WriteResult` with three fields:
```python
@dataclass
class WriteResult:
    error: str | None = None        # Error message on failure
    path: str | None = None          # Path of written file
    files_update: dict | None = None  # State update dict (or None)
```

The `files_update` field is what controls the branching:
- **`StateBackend`**: Returns `files_update={path: file_data}` → middleware wraps this in `Command(update={"files": ...})`
- **`FilesystemBackend`**: Returns `files_update=None` → file is already on disk, middleware just returns a string confirmation
- **`StoreBackend`**: Returns `files_update=None` → file is already in the Store, no state update needed

This is the actual code from `FilesystemMiddleware._create_write_file_tool()`:
```python
res: WriteResult = backend.write(file_path, content)
if res.files_update is not None:
    # StateBackend: update LangGraph state
    return Command(update={"files": res.files_update, "messages": [...]})
# FilesystemBackend/StoreBackend: already persisted
return f"Updated file {res.path}"
```

In [ ]:
# Let's see the difference in practice
from deepagents.backends.protocol import WriteResult

# Simulate what each backend returns for the same write() call

# StateBackend: "here's the data, put it in state"
state_result = WriteResult(
    path="/notes.md",
    files_update={"/notes.md": {"content": ["hello"], "created_at": "...", "modified_at": "..."}}
)
print(f"StateBackend:      files_update = {type(state_result.files_update).__name__} (middleware issues Command)")

# FilesystemBackend: "I already wrote it to disk"
fs_result = WriteResult(path="/notes.md", files_update=None)
print(f"FilesystemBackend: files_update = {fs_result.files_update} (middleware returns string)")

# StoreBackend: "I already put it in the Store" 
store_result = WriteResult(path="/notes.md", files_update=None)
print(f"StoreBackend:      files_update = {store_result.files_update} (middleware returns string)")

### StateBackend — Ephemeral, in LangGraph state

This is the default. Files live in `runtime.state["files"]` — a dict managed by the LangGraph checkpointer.

```python
class StateBackend(BackendProtocol):
    def __init__(self, runtime: ToolRuntime):
        self.runtime = runtime
    
    def read(self, file_path, ...):
        files = self.runtime.state.get("files", {})
        return files[file_path]  # Read directly from state
    
    def write(self, file_path, content):
        file_data = create_file_data(content)
        # Return the data so middleware can update state via Command
        return WriteResult(path=file_path, files_update={file_path: file_data})
```

**Lifecycle:** Files persist within a thread (via checkpointer) but vanish across threads.

**Trade-off:** Fast (in-memory), but every file is serialized into the checkpoint. Large files bloat checkpoint size.

In [ ]:
# StateBackend in action — files live in state["files"]
from deepagents import create_deep_agent
from deepagents.backends import StateBackend

state_agent = create_deep_agent(
    model=model,
    backend=StateBackend,  # Default — can pass the class itself (factory)
    system_prompt="You are a helpful assistant.",
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = state_agent.invoke({
    "messages": [{"role": "user", "content": "Write a file /demo.txt with 'hello from StateBackend'"}]
}, config=config)

# Files are in the result state
print("Files in state:", list(result.get("files", {}).keys()))
if "/demo.txt" in result.get("files", {}):
    print("Content:", result["files"]["/demo.txt"]["content"])

### FilesystemBackend — Real disk I/O

Reads and writes actual files on disk. The key parameter is `virtual_mode`:

- `virtual_mode=True`: Paths are sandboxed under `root_dir`. Agent writes to `/notes.md` → actually writes to `{root_dir}/notes.md`. Path traversal (`..`) is blocked.
- `virtual_mode=False` (default, deprecated): Absolute paths work as-is. **No security boundary.**

```python
class FilesystemBackend(BackendProtocol):
    def __init__(self, root_dir, virtual_mode=True):
        self.cwd = Path(root_dir).resolve()
        self.virtual_mode = virtual_mode
    
    def write(self, file_path, content):
        resolved = self._resolve_path(file_path)  # Sandboxing happens here
        resolved.parent.mkdir(parents=True, exist_ok=True)
        resolved.write_text(content)
        return WriteResult(path=file_path, files_update=None)  # Already on disk!
```

**Key difference from StateBackend:** `files_update=None` — the file is already persisted to disk. The middleware returns a plain string response, no `Command` needed.

**Lifecycle:** Files persist on disk permanently. No checkpointer needed for persistence, but files are shared across all threads that use the same `root_dir`.

In [ ]:
# FilesystemBackend in action — files go to real disk
import tempfile, os
from deepagents.backends import FilesystemBackend

sandbox_dir = tempfile.mkdtemp(prefix="deepagents_301_")
print(f"Sandbox directory: {sandbox_dir}")

fs_agent = create_deep_agent(
    model=model,
    backend=FilesystemBackend(root_dir=sandbox_dir, virtual_mode=True),
    system_prompt="You are a helpful assistant.",
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = fs_agent.invoke({
    "messages": [{"role": "user", "content": "Write a file /demo.txt with 'hello from FilesystemBackend'"}]
}, config=config)

# No files in state — they're on disk!
print(f"Files in state: {list(result.get('files', {}).keys())}")

# But they exist on disk
disk_path = os.path.join(sandbox_dir, "demo.txt")
if os.path.exists(disk_path):
    print(f"File on disk: {open(disk_path).read()}")

# Cleanup
import shutil
shutil.rmtree(sandbox_dir, ignore_errors=True)

### StoreBackend — Persistent, cross-thread storage

Uses LangGraph's `BaseStore` (a key-value store). Files persist across all threads — write in thread A, read in thread B.

```python
class StoreBackend(BackendProtocol):
    def __init__(self, runtime: ToolRuntime, *, namespace=None):
        self.runtime = runtime
    
    def write(self, file_path, content):
        store = self.runtime.store          # LangGraph provides this
        namespace = self._get_namespace()    # e.g., ("filesystem",)
        file_data = create_file_data(content)
        store.put(namespace, file_path, file_data)  # Persisted immediately
        return WriteResult(path=file_path, files_update=None)  # Already in store!
```

**Namespace scoping** controls isolation:
- `("filesystem",)` — all threads/users share files
- `(user_id, "filesystem")` — per-user isolation
- `(assistant_id, "filesystem")` — per-assistant isolation

**Lifecycle:** Files persist as long as the Store exists. In LangGraph Platform, this means across deployments.

In [ ]:
# StoreBackend in action — files persist across threads
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend

store = InMemoryStore()

store_agent = create_deep_agent(
    model=model,
    backend=lambda rt: StoreBackend(rt, namespace=lambda ctx: ("filesystem",)),
    store=store,
    system_prompt="You are a helpful assistant.",
    checkpointer=MemorySaver(),
)

# Thread 1: Write a file
config_t1 = {"configurable": {"thread_id": str(uuid7())}}
result = store_agent.invoke({
    "messages": [{"role": "user", "content": "Write a file /shared_note.txt with 'This persists across threads!'"}]
}, config=config_t1)
print("Thread 1:", result["messages"][-1].content)

# Thread 2: Read it (different thread!)
config_t2 = {"configurable": {"thread_id": str(uuid7())}}
result = store_agent.invoke({
    "messages": [{"role": "user", "content": "Read the file /shared_note.txt"}]
}, config=config_t2)
print("Thread 2:", result["messages"][-1].content)

### Backend comparison

| | StateBackend | FilesystemBackend | StoreBackend |
|---|---|---|---|
| **Storage** | LangGraph state (checkpoint) | Real filesystem | LangGraph Store (key-value) |
| **Persistence** | Within thread | Permanent on disk | Cross-thread, cross-session |
| **`files_update`** | `{path: data}` → `Command` | `None` → string | `None` → string |
| **State bloat** | Yes (files in checkpoint) | No | No |
| **Use case** | Scratch space, notebooks | CLI tools, local dev | LangGraph Platform, production |
| **Needs `store=`** | No | No | Yes (`InMemoryStore` or platform store) |

In production, you typically use `CompositeBackend` to mix them — ephemeral scratch space via `StateBackend`, persistent memories via `StoreBackend`:

```python
backend = lambda rt: CompositeBackend(
    default=StateBackend(rt),           # Scratch files go to state
    routes={
        "/memories/": StoreBackend(rt),  # Memories persist in Store
    },
)
```

### Key Takeaway
- The `WriteResult.files_update` field controls whether the middleware issues a `Command` or returns a string
- `StateBackend` returns data for state updates; `FilesystemBackend` and `StoreBackend` persist directly and return `None`
- All three backends implement the same `BackendProtocol` — tools don't know which one they're using
- `CompositeBackend` routes different paths to different backends for hybrid storage

---
## Part 4: Custom Task Definitions for Subagents

By default, `SubAgentMiddleware` creates a `task()` tool with two parameters:
- `description` (str) — a raw string describing what to do
- `subagent_type` (str) — which subagent to invoke

The subagent receives `HumanMessage(content=description)` — just a flat string.

This works, but has limitations:
- No structure — the LLM has to pack everything into prose
- No type safety — no validation on what gets passed
- No selective context — the subagent gets all non-excluded state

Let's build custom task tools that fix these issues.

In [ ]:
# First, let's see what the default task tool does
# Here's the core of _build_task_tool (simplified):

from deepagents.middleware.subagents import _EXCLUDED_STATE_KEYS
print("State keys excluded from subagents:", _EXCLUDED_STATE_KEYS)

# The default flow:
# 1. Main agent calls: task(description="Research LangGraph", subagent_type="researcher")
# 2. SubAgentMiddleware:
#    a. Filters state: {k: v for k, v in state if k not in EXCLUDED_STATE_KEYS}
#    b. Sets messages: [HumanMessage("Research LangGraph")]  <-- just the raw string
#    c. Invokes subagent
#    d. Returns: Command(update={"messages": [ToolMessage(result)]})

In [ ]:
# Custom Task Definition 1: Structured Input
#
# Instead of a raw string, define a Pydantic model for the task input.
# This gives the orchestrator agent a clear schema to fill out.

from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, AIMessage


class ResearchTask(BaseModel):
    """A structured research task definition."""
    topic: str = Field(description="The topic to research")
    depth: str = Field(description="'shallow' (quick overview), 'medium' (key details), or 'deep' (comprehensive)")
    output_format: str = Field(description="'bullet_points', 'narrative', or 'structured_report'")
    max_sources: int = Field(default=3, description="Maximum number of sources to consult")
    focus_areas: list[str] = Field(default_factory=list, description="Specific aspects to focus on")


def build_structured_task_tool(subagent_graph):
    """Build a task tool with structured input instead of a raw string."""
    
    def research(
        topic: Annotated[str, "The topic to research"],
        depth: Annotated[str, "'shallow', 'medium', or 'deep'"],
        output_format: Annotated[str, "'bullet_points', 'narrative', or 'structured_report'"],
        max_sources: Annotated[int, "Maximum sources to consult"] = 3,
        focus_areas: Annotated[list[str], "Specific aspects to focus on"] = [],
        runtime: ToolRuntime = None,
    ) -> Command:
        """Delegate a structured research task to the research subagent."""
        # Build a well-structured prompt from the typed fields
        focus_str = "\n".join(f"  - {area}" for area in focus_areas) if focus_areas else "  (none specified)"
        
        prompt = f"""## Research Task

**Topic:** {topic}
**Depth:** {depth}
**Max Sources:** {max_sources}

**Focus Areas:**
{focus_str}

**Output Format:** {output_format}

Conduct the research and return your findings in the specified format."""
        
        # Filter state (same pattern as SubAgentMiddleware)
        subagent_state = {
            k: v for k, v in runtime.state.items()
            if k not in _EXCLUDED_STATE_KEYS
        }
        subagent_state["messages"] = [HumanMessage(content=prompt)]
        
        result = subagent_graph.invoke(subagent_state)
        
        return Command(update={
            "messages": [ToolMessage(
                result["messages"][-1].text.rstrip(),
                tool_call_id=runtime.tool_call_id,
            )],
        })
    
    return StructuredTool.from_function(
        name="research",
        func=research,
        description="Delegate a structured research task to a specialist subagent.",
    )


print("Structured task tool builder defined")

In [ ]:
# Create the research subagent and the orchestrator

# Subagent: a simple create_agent with search capabilities
research_subagent = create_agent(
    model,
    tools=[tavily_search],
    system_prompt="You are a research specialist. Follow the task instructions precisely.",
    middleware=[
        PatchToolCallsMiddleware(),
    ],
)

# Build the structured task tool
structured_research_tool = build_structured_task_tool(research_subagent)

# Orchestrator: uses the structured task tool
orchestrator = create_agent(
    model,
    tools=[structured_research_tool],
    system_prompt="""You are a research coordinator. When asked to research something:
1. Use the `research` tool to delegate to a specialist
2. Always specify depth, output_format, and relevant focus_areas
3. Summarize the results for the user""",
    middleware=[
        TodoListMiddleware(),
        FilesystemMiddleware(backend=StateBackend),
        PatchToolCallsMiddleware(),
    ],
).with_config({"recursion_limit": 100})

print("Orchestrator ready with structured research tool")

In [ ]:
# Test: The orchestrator now passes structured fields, not raw strings
config = {"configurable": {"thread_id": str(uuid7())}}

result = orchestrator.invoke({
    "messages": [{"role": "user", "content": "Research what's new with LangGraph in 2025. Focus on the agent framework and middleware system. Give me bullet points."}]
}, config=config)

print(result["messages"][-1].content[:2000])

### Custom Task Definition 2: Context-Aware Delegation

The default `task()` passes **all** non-excluded state to subagents. Sometimes you want to be selective — pass only specific files, or inject additional context.

In [ ]:
# Context-aware task tool: selectively pass files to the subagent

def build_context_aware_task_tool(subagent_graph):
    """Task tool that lets the orchestrator choose which files to share."""
    
    def research_with_context(
        description: Annotated[str, "Detailed task description"],
        include_files: Annotated[list[str], "File paths from the filesystem to pass to the subagent"] = [],
        additional_context: Annotated[str, "Extra context to prepend to the task"] = "",
        runtime: ToolRuntime = None,
    ) -> Command:
        """Delegate a task with selective file context."""
        # Only pass explicitly requested files
        all_files = runtime.state.get("files", {})
        selected_files = {k: v for k, v in all_files.items() if k in include_files}
        
        # Build the task message
        parts = []
        if additional_context:
            parts.append(f"## Context\n{additional_context}")
        if selected_files:
            parts.append("## Reference Files")
            for path, data in selected_files.items():
                content = "\n".join(data.get("content", []))
                parts.append(f"### `{path}`\n```\n{content}\n```")
        parts.append(f"## Task\n{description}")
        
        prompt = "\n\n".join(parts)
        
        # Invoke with clean state + selected files only
        subagent_state = {
            "messages": [HumanMessage(content=prompt)],
            "files": selected_files,
        }
        
        result = subagent_graph.invoke(subagent_state)
        
        return Command(update={
            "messages": [ToolMessage(
                result["messages"][-1].text.rstrip(),
                tool_call_id=runtime.tool_call_id,
            )],
        })
    
    return StructuredTool.from_function(
        name="research_with_context",
        func=research_with_context,
        description="Delegate a research task with selective file context.",
    )


print("Context-aware task tool defined")
print("Key difference: subagent only sees files you explicitly pass")

### Why custom task tools matter

| Approach | Pros | Cons |
|----------|------|------|
| Default `task(description=...)` | Generic, works for everything | No structure, LLM packs everything into prose |
| Structured fields | Type-safe, clear expectations | One tool per subagent type |
| Context-aware | Selective file sharing, clean context | More orchestrator decision-making required |

In practice, you'll often use **one custom tool per subagent** rather than the generic `task()` tool — especially when the subagent has a well-defined interface.

### Key Takeaway
- The default `task()` tool passes a raw string as `HumanMessage` — you can replace this with structured Pydantic fields
- `_EXCLUDED_STATE_KEYS` controls what state leaks to subagents — customize this for your needs
- Custom task tools give you control over exactly what context the subagent sees

---
## Part 5: StateGraph Workflows as Subagents

Subagents don't have to be agents (LLM loops). You can plug in any `Runnable` that has `messages` in its output — including a deterministic `StateGraph` workflow.

This is powerful when you have a **fixed pipeline** (gather → analyze → format) that shouldn't be left to an LLM to orchestrate.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# Define the workflow state — must include `messages` for CompiledSubAgent
class ResearchWorkflowState(TypedDict):
    messages: list          # Required: CompiledSubAgent reads the last message
    topic: str              # Extracted from the incoming HumanMessage
    raw_data: str           # Output of the gather step
    analysis: str           # Output of the analyze step


# Step 1: Extract the topic and gather data
def gather_data(state: ResearchWorkflowState):
    """Search for information on the topic."""
    # The task tool sends HumanMessage(description) -- extract the topic
    topic = state["messages"][-1].content
    
    # Use tavily directly (not as a tool -- this is a deterministic step)
    results = tavily_client.search(topic, max_results=3)
    raw = "\n\n".join(
        f"Source: {r['url']}\n{r['content']}" 
        for r in results["results"]
    )
    return {"topic": topic, "raw_data": raw}


# Step 2: LLM analysis of the raw data
def analyze_data(state: ResearchWorkflowState):
    """Have the LLM analyze and synthesize the gathered data."""
    response = model.invoke([
        {"role": "system", "content": "Analyze the following research data. Identify key themes, facts, and insights. Be concise."},
        {"role": "user", "content": f"Topic: {state['topic']}\n\nData:\n{state['raw_data']}"}
    ])
    return {"analysis": response.content}


# Step 3: Format the final report and set messages for the parent agent
def format_report(state: ResearchWorkflowState):
    """Format into a final report and return via messages."""
    response = model.invoke([
        {"role": "system", "content": "Write a concise research summary with bullet points based on this analysis."},
        {"role": "user", "content": state["analysis"]}
    ])
    # The last message is what gets returned to the parent agent
    return {"messages": [AIMessage(content=response.content)]}


# Build the workflow graph
workflow = StateGraph(ResearchWorkflowState)
workflow.add_node("gather", gather_data)
workflow.add_node("analyze", analyze_data)
workflow.add_node("format", format_report)
workflow.add_edge(START, "gather")
workflow.add_edge("gather", "analyze")
workflow.add_edge("analyze", "format")
workflow.add_edge("format", END)

compiled_workflow = workflow.compile()
print("Research workflow compiled: gather → analyze → format")

In [ ]:
# Test the workflow standalone first
standalone_result = compiled_workflow.invoke({
    "messages": [HumanMessage(content="Latest developments in AI agents 2025")]
})

print("Workflow output (last message):")
print(standalone_result["messages"][-1].content[:1000])

In [ ]:
# Now plug it in as a CompiledSubAgent
from deepagents import create_deep_agent

agent_with_workflow = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt="You are a research coordinator. Use the research-pipeline for structured research.",
    subagents=[
        {
            # CompiledSubAgent: any Runnable with 'messages' in state
            "name": "research-pipeline",
            "description": "A deterministic research pipeline that gathers data, analyzes it, and produces a structured report. Use this for any research request.",
            "runnable": compiled_workflow,
        },
    ],
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = agent_with_workflow.invoke({
    "messages": [{"role": "user", "content": "Research the current state of LangGraph's middleware system"}]
}, config=config)

print(result["messages"][-1].content[:2000])

### Agent subagent vs. Workflow subagent

| | Agent subagent (default) | Workflow subagent |
|---|---|---|
| **Execution** | LLM decides steps | Fixed pipeline |
| **Flexibility** | Can adapt on the fly | Deterministic |
| **Cost** | More tokens (LLM reasoning) | Less tokens (structured steps) |
| **Debugging** | Harder (LLM choices vary) | Easier (fixed path) |
| **When to use** | Open-ended tasks | Well-defined pipelines |

The `CompiledSubAgent` contract is simple: your `Runnable` must have `messages` in its output state. The last message is returned to the parent as a `ToolMessage`.

### Key Takeaway
- Any `Runnable` with `messages` in state can be a subagent via `CompiledSubAgent`
- Deterministic workflows are cheaper and more predictable than agent subagents
- Mix both: use agent subagents for open-ended tasks, workflow subagents for pipelines

---
## Part 6: MCP Adapters for Tools

[Model Context Protocol (MCP)](https://modelcontextprotocol.io/) provides a standard way to expose tools from external servers. The key insight: **MCP tools become regular LangChain `BaseTool` objects**, so they're interchangeable with any other tool.

### How MCP tool adapters work

```
┌─────────────┐     ┌───────────────┐     ┌─────────────┐
│  MCP Server  │────▶│  MCP Adapter   │────▶│  BaseTool    │
│ (external)   │     │ (langchain-mcp)│     │ (standard)   │
└─────────────┘     └───────────────┘     └─────────────┘
                                                │
                    ┌───────────────────────────┘
                    ▼
        create_agent(tools=[mcp_tool, regular_tool])
```

1. **Connect** to an MCP server (stdio or HTTP/SSE)
2. **Discover** tools via `server.list_tools()`
3. **Wrap** each MCP tool as a `StructuredTool` that calls `server.call_tool()` on invocation
4. **Use** them like any other LangChain tool

In [ ]:
# The MCP adapter pattern (what langchain-mcp-adapters does under the hood)
#
# This is pseudocode to illustrate the concept -- the actual implementation
# handles JSON schema conversion, async, and error handling.

def mcp_tool_to_langchain(mcp_server, mcp_tool_spec):
    """Convert an MCP tool spec into a LangChain StructuredTool.
    
    This is conceptually what langchain-mcp-adapters does:
    1. Take the tool's name, description, and JSON schema from MCP
    2. Create a StructuredTool that calls the MCP server on invocation
    """
    def invoke_mcp_tool(**kwargs):
        # On each call, forward to the MCP server
        result = mcp_server.call_tool(mcp_tool_spec["name"], kwargs)
        return result.content
    
    return StructuredTool.from_function(
        name=mcp_tool_spec["name"],
        func=invoke_mcp_tool,
        description=mcp_tool_spec["description"],
        # args_schema would be derived from MCP's inputSchema
    )

print("MCP adapter pattern: MCP tool spec → StructuredTool → works anywhere")

In [ ]:
# Using MCP tools with create_deep_agent
#
# Install: pip install langchain-mcp-adapters mcp
#
# Example with a filesystem MCP server:

try:
    from langchain_mcp_adapters.client import MultiServerMCPClient
    
    print("langchain-mcp-adapters is available!")
    print()
    print("Example usage:")
    print("""
    async with MultiServerMCPClient({
        "filesystem": {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-filesystem", "/tmp/workspace"],
            "transport": "stdio",
        },
        "web-search": {
            "url": "http://localhost:8080/sse",
            "transport": "sse",
        },
    }) as client:
        mcp_tools = client.get_tools()  # Returns list[BaseTool]
        
        # Mix MCP tools with regular tools -- they're all BaseTool
        agent = create_deep_agent(
            model=model,
            tools=[tavily_search, *mcp_tools],
        )
    """)
    
except ImportError:
    print("langchain-mcp-adapters not installed.")
    print("Install with: pip install langchain-mcp-adapters mcp")
    print()
    print("The key point: MCP tools become standard BaseTool objects.")
    print("Once adapted, they work identically to @tool-decorated functions.")
    print("You can pass them to create_agent, create_deep_agent, or any LangChain agent.")

### When MCP changes your invocation pattern

MCP tools are **remote procedure calls**. This has practical implications:

| Aspect | Native `@tool` | MCP Tool |
|--------|----------------|----------|
| Latency | In-process | Network roundtrip |
| State access | `ToolRuntime` available | No state access (stateless RPC) |
| Error handling | Python exceptions | MCP error protocol |
| Lifecycle | Lives with the agent | Server must be running |

The most important difference: **MCP tools can't access `ToolRuntime`**. They can't read agent state, return `Command` objects, or modify state directly. They're pure input → output functions.

This means you can't replace `FilesystemMiddleware` tools with MCP filesystem tools and get the same state management. MCP tools are best for **external capabilities** (web search, databases, APIs) rather than state-coupled operations.

### Key Takeaway
- MCP tools become standard `BaseTool` objects via adapters
- Mix MCP and native tools freely in `create_agent` / `create_deep_agent`
- MCP tools are stateless RPCs — they can't access agent state or return `Command` objects

---
## Part 7: Sandbox Usage

Deep Agents can execute shell commands when the backend implements `SandboxBackendProtocol`. This adds an `execute` tool to the agent.

### The sandbox backend hierarchy

```
BackendProtocol          ← File operations only (ls, read, write, edit, glob, grep)
  └── SandboxBackendProtocol  ← + execute(command, timeout) + id property
        └── BaseSandbox       ← Implements file ops via execute() (shell commands)
              └── LangSmithSandbox  ← Wraps langsmith.sandbox.Sandbox
```

`BaseSandbox` is the interesting layer: it implements **all** file operations by shelling out commands via `execute()`. So `read("/file.txt")` internally runs something like `python3 -c "import base64; print(base64.b64encode(open('/file.txt','rb').read()).decode())"` inside the sandbox.

This means a sandbox backend only needs to implement `execute()` — everything else is derived from it.

In [ ]:
# Let's look at the protocol
from deepagents.backends.protocol import BackendProtocol, SandboxBackendProtocol, ExecuteResponse

# BackendProtocol methods (file operations):
backend_methods = [m for m in dir(BackendProtocol) if not m.startswith('_') and not m.startswith('a')]
print("BackendProtocol methods:", backend_methods)

# SandboxBackendProtocol adds:
sandbox_only = [m for m in dir(SandboxBackendProtocol) if not m.startswith('_') and not m.startswith('a') and m not in dir(BackendProtocol)]
print("SandboxBackendProtocol adds:", sandbox_only)

# The execute response is simple:
print(f"\nExecuteResponse fields: output (str), exit_code (int|None), truncated (bool)")

In [ ]:
# Using LangSmithSandbox with create_deep_agent
#
# LangSmithSandbox wraps the LangSmith SDK's Sandbox class.
# This gives you a cloud-hosted sandbox environment for code execution.

import os

if os.environ.get("LANGSMITH_API_KEY"):
    try:
        from langsmith.sandbox import Sandbox
        from deepagents.backends.langsmith import LangSmithSandbox
        
        # Create a LangSmith sandbox
        sandbox = Sandbox()
        backend = LangSmithSandbox(sandbox)
        
        print(f"Sandbox ID: {backend.id}")
        
        # Test direct execution
        result = backend.execute("echo 'Hello from sandbox!' && python3 --version")
        print(f"Output: {result.output}")
        print(f"Exit code: {result.exit_code}")
        
        # Create an agent with sandbox capabilities
        sandbox_agent = create_deep_agent(
            model=model,
            tools=[tavily_search],
            backend=backend,  # This enables the `execute` tool
            system_prompt="You are a coding assistant with sandbox access.",
        )
        
        config = {"configurable": {"thread_id": str(uuid7())}}
        result = sandbox_agent.invoke({
            "messages": [{"role": "user", "content": "Write a Python script that prints the first 10 fibonacci numbers, save it, and run it."}]
        }, config=config)
        print("\n" + result["messages"][-1].content)
        
    except Exception as e:
        print(f"Sandbox error: {e}")
        print("Make sure your LangSmith plan supports sandboxes.")
else:
    print("LANGSMITH_API_KEY not set — skipping sandbox demo.")
    print()
    print("To use sandboxes:")
    print("1. Set LANGSMITH_API_KEY in your .env")
    print("2. The sandbox backend adds an `execute` tool automatically")
    print("3. File operations go through shell commands inside the sandbox")
    print("4. Your agent can write code, run it, and see the output")

In [ ]:
# How FilesystemMiddleware decides whether to add the execute tool:

from deepagents.middleware.filesystem import FilesystemMiddleware

# With StateBackend (no sandbox) -- no execute tool
fs_no_sandbox = FilesystemMiddleware(backend=StateBackend)
print("StateBackend tools:", [t.name for t in fs_no_sandbox.tools])

# The check is simple: does the backend implement SandboxBackendProtocol?
# If yes → execute tool is added
# If no → only file operation tools

### Key Takeaway
- `SandboxBackendProtocol` adds `execute(command, timeout)` to the backend interface
- `BaseSandbox` implements all file operations as shell commands via `execute()` — so you only need to implement one method
- `LangSmithSandbox` gives you cloud-hosted code execution
- The `execute` tool is **only** available when the backend supports it (checked at middleware init)

---
## Part 8: Context Management Failures

Deep Agents uses a multi-layer context management strategy:

1. **Argument truncation** (lightweight) — truncate large tool call args in old messages
2. **Full summarization** (heavy) — LLM summarizes old messages, offloads to filesystem
3. **Tool result eviction** — huge tool outputs are saved to `/large_tool_results/` and replaced with a pointer

When any of these fail, things get interesting. Let's see what happens.

In [ ]:
# First, let's see the default summarization settings for our model
from deepagents.middleware.summarization import compute_summarization_defaults

defaults = compute_summarization_defaults(model)
print("Summarization defaults for your model:")
print(f"  Trigger: {defaults['trigger']}")
print(f"  Keep:    {defaults['keep']}")
print(f"  Truncate args: {defaults['truncate_args_settings']}")

# What these mean:
# trigger=("fraction", 0.85) → summarize when 85% of context window is used
# keep=("fraction", 0.10)   → keep only 10% of messages after summarization
# truncate_args trigger at same thresholds → truncate large tool args first

### Scenario 1: Watching Summarization Trigger

Let's set an artificially low trigger to watch summarization happen in real-time.

In [ ]:
from langchain.agents.middleware.summarization import SummarizationMiddleware as LCSummarizationMiddleware

# Create an agent with an aggressive summarization trigger
low_trigger_agent = create_agent(
    model,
    tools=[tavily_search],
    system_prompt="You are a helpful research assistant. Be concise.",
    middleware=[
        TodoListMiddleware(),
        FilesystemMiddleware(backend=StateBackend),
        LCSummarizationMiddleware(
            model=model,
            trigger=("messages", 8),   # Summarize after just 8 messages!
            keep=("messages", 4),      # Keep only the last 4
        ),
        PatchToolCallsMiddleware(),
    ],
    checkpointer=MemorySaver(),
)

config = {"configurable": {"thread_id": str(uuid7())}}

# Turn 1: Normal response
result = low_trigger_agent.invoke({
    "messages": [{"role": "user", "content": "What is LangGraph?"}]
}, config=config)
msg_count = len(result["messages"])
print(f"Turn 1: {msg_count} messages in state")
print(f"Response: {result['messages'][-1].content[:200]}...")

In [ ]:
# Turn 2: This might trigger summarization (agent + tool call messages add up)
result = low_trigger_agent.invoke({
    "messages": [{"role": "user", "content": "Search the web and tell me the latest news about it."}]
}, config=config)
msg_count = len(result["messages"])
print(f"Turn 2: {msg_count} messages in state")

# Check if summarization happened -- look for summary messages
for i, msg in enumerate(result["messages"][:3]):
    content_preview = msg.content[:150] if isinstance(msg.content, str) else str(msg.content)[:150]
    print(f"  msg[{i}] ({type(msg).__name__}): {content_preview}...")

In [ ]:
# Turn 3: By now summarization should definitely have fired
result = low_trigger_agent.invoke({
    "messages": [{"role": "user", "content": "Now search for information about Deep Agents and compare with what you told me about LangGraph."}]
}, config=config)
msg_count = len(result["messages"])
print(f"Turn 3: {msg_count} messages in state")
print()

# After summarization, early messages are replaced with a summary
print("First message in state (may be a summary):")
first = result["messages"][0]
content = first.content if isinstance(first.content, str) else str(first.content)
print(f"  Type: {type(first).__name__}")
print(f"  Content: {content[:500]}")

### Scenario 2: What happens with very large tool results

When a tool returns a massive result, the summarization middleware's argument truncation kicks in on subsequent turns — it truncates the large args in older messages.

In [ ]:
@tool
def verbose_tool(query: str) -> str:
    """A tool that returns an extremely large result to demonstrate context pressure."""
    # Simulate a tool that returns way too much data
    return f"Results for '{query}':\n" + ("This is a very detailed finding. " * 500)

# Create an agent with both summarization and the verbose tool
from deepagents.middleware.summarization import _DeepAgentsSummarizationMiddleware

context_pressure_agent = create_agent(
    model,
    tools=[verbose_tool],
    system_prompt="You are a test assistant. Use verbose_tool when asked to search.",
    middleware=[
        _DeepAgentsSummarizationMiddleware(
            model=model,
            backend=StateBackend,
            trigger=("messages", 10),
            keep=("messages", 4),
            truncate_args_settings={
                "trigger": ("messages", 6),   # Truncate args after 6 messages
                "keep": ("messages", 4),       # Leave last 4 untouched
                "max_length": 100,             # Truncate args longer than 100 chars
                "truncation_text": "...(truncated for context management)",
            },
        ),
        PatchToolCallsMiddleware(),
    ],
    checkpointer=MemorySaver(),
)

config = {"configurable": {"thread_id": str(uuid7())}}

# Turn 1: Trigger the verbose tool
result = context_pressure_agent.invoke({
    "messages": [{"role": "user", "content": "Search for information about Python."}]
}, config=config)
print(f"Turn 1: {len(result['messages'])} messages")
print(f"Response: {result['messages'][-1].content[:200]}")

In [ ]:
# Turn 2: More pressure
result = context_pressure_agent.invoke({
    "messages": [{"role": "user", "content": "Now search for JavaScript too."}]
}, config=config)
print(f"Turn 2: {len(result['messages'])} messages")

# Turn 3: By now, arg truncation should have fired on the older tool calls
result = context_pressure_agent.invoke({
    "messages": [{"role": "user", "content": "Compare what you found about Python and JavaScript."}]
}, config=config)
print(f"Turn 3: {len(result['messages'])} messages")
print(f"\nAgent can still answer (summarization preserved key info):")
print(result["messages"][-1].content[:500])

### Scenario 3: ContextOverflowError — when everything fails

If the context fills up faster than summarization can compact it (or if summarization is disabled), you'll hit `ContextOverflowError`. This is an error from the **model provider** rejecting the request.

In [ ]:
from langchain_core.exceptions import ContextOverflowError

# ContextOverflowError is what you'd see if:
# 1. Summarization is disabled/not configured
# 2. Summarization can't keep up with context growth
# 3. A single message is larger than the context window

print("ContextOverflowError scenarios and recovery strategies:")
print()
print("1. No summarization configured:")
print("   → Add SummarizationMiddleware with appropriate triggers")
print()
print("2. Trigger too high (summarization fires too late):")
print("   → Lower the trigger: ('fraction', 0.7) instead of ('fraction', 0.85)")
print()
print("3. Single massive tool result fills context:")
print("   → Use tool_result eviction (FilesystemMiddleware does this automatically)")
print("   → Or limit tool output size in your tool implementation")
print()
print("4. Summary itself is too large:")
print("   → Reduce trim_tokens_to_summarize (default 4000)")
print("   → Use fraction-based keep instead of message-count keep")

In [ ]:
# Defensive pattern: wrap agent invocation with context overflow handling

def invoke_with_recovery(agent, input_msg, config, max_retries=1):
    """Invoke an agent with context overflow recovery."""
    try:
        return agent.invoke(input_msg, config=config)
    except ContextOverflowError as e:
        print(f"⚠️  Context overflow detected: {str(e)[:200]}")
        
        if max_retries > 0:
            print("Attempting recovery: starting fresh thread with summary...")
            
            # Recovery: start a new thread with a summary of what happened
            new_config = {"configurable": {"thread_id": str(uuid7())}}
            recovery_msg = {
                "messages": [{
                    "role": "user",
                    "content": f"Previous conversation overflowed. Original request: {input_msg['messages'][-1]['content'] if isinstance(input_msg['messages'][-1], dict) else input_msg['messages'][-1].content}"
                }]
            }
            return invoke_with_recovery(agent, recovery_msg, new_config, max_retries - 1)
        
        raise

print("Recovery pattern defined")
print("In production, you'd also want to:")
print("- Log the overflow event")
print("- Alert on repeated overflows (indicates misconfigured thresholds)")
print("- Consider using the SummarizationToolMiddleware for on-demand compaction")

### Context management tuning guide

| Setting | Conservative | Default | Aggressive |
|---------|-------------|---------|------------|
| `trigger` | `("fraction", 0.6)` | `("fraction", 0.85)` | `("fraction", 0.95)` |
| `keep` | `("fraction", 0.05)` | `("fraction", 0.10)` | `("fraction", 0.20)` |
| `truncate_args trigger` | `("fraction", 0.5)` | Same as trigger | Disabled |
| `max_length` (arg truncation) | `500` | `2000` | `5000` |

**Rules of thumb:**
- If agents hit overflow, lower the trigger
- If agents lose too much context, raise the keep
- If costs are too high, lower both trigger and keep
- Always enable arg truncation — it's much cheaper than full summarization

### Key Takeaway
- Context management has three layers: arg truncation → full summarization → tool result eviction
- `compute_summarization_defaults(model)` shows what your model gets automatically
- `ContextOverflowError` means the model rejected the request — fix with lower triggers
- The `SummarizationToolMiddleware` lets agents trigger compaction themselves via a `compact_conversation` tool

---
## Summary

| Part | What you learned | Key pattern |
|------|-----------------|-------------|
| **1. Deconstructing** | `create_deep_agent` = `create_agent` + middleware | Middleware stack composition |
| **2. Filesystem** | Tools return `Command(update={...})` to modify state | State reducers + Command pattern |
| **3. Backends** | `WriteResult.files_update` controls state vs. direct persistence | Backend abstraction, `CompositeBackend` |
| **4. Task definitions** | Replace raw string tasks with structured Pydantic input | Custom `StructuredTool` per subagent |
| **5. Workflow subagents** | Any `Runnable` with `messages` can be a subagent | `CompiledSubAgent` contract |
| **6. MCP adapters** | MCP tools become standard `BaseTool` objects | Adapter pattern, stateless RPC |
| **7. Sandboxes** | `SandboxBackendProtocol` gates the `execute` tool | Protocol extension pattern |
| **8. Context failures** | Three-layer context management, recovery strategies | Defensive invocation, tuning |

### Where to go from here

- **Build your own middleware** — now that you understand the hooks, create middleware for logging, caching, rate limiting, or custom behavior
- **Custom backends** — implement `BackendProtocol` for your storage (S3, database, etc.)
- **Deploy with LangGraph Platform** — use `langgraph dev` to run your agents with a Studio UI
- **Evaluate** — use LangSmith to measure agent quality with datasets and evaluators